# 🏥 Lab 3: Medical Image Classification with Neural Networks
# **SOLUTION NOTEBOOK**

**AI in Medicine and Healthcare**  
**Insper Instituto de Ensino e Pesquisa**  

## 📚 Pedagogical Notes

**Learning Goals:**
1. Apply NN knowledge from Lab 2 to new domain
2. **Experience productive struggle** with MLPs on images
3. Build intuition for why CNNs are needed
4. Create motivation for next week's DICOM/imaging content

**Expected Results:**
- Accuracy: 60-70% (deliberately mediocre!)
- Training: Slow compared to Lab 2
- Student reaction: "Wait.. This works better... what's wrong?"

**Key Teaching Moment:**
- MLP must be finding a shortcut
- More parameters doesn't always help
- Flattening destroys spatial information
- Architecture matters!

---

In [ ]:
# SOLUTION: Complete imports
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision
from torchvision import transforms
from PIL import Image

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, recall_score, precision_score

import os
from pathlib import Path
import time

# Set random seeds
torch.manual_seed(42)
np.random.seed(42)

print(f"PyTorch version: {torch.__version__}")
print(f"Torchvision version: {torchvision.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

# PART 1: Image Loading & Visualization - SOLUTION

In [ ]:
# Download and extract dataset
# Replace FILE_ID with your Google Drive file ID
!gdown --id 17MSlc-0l_SSRevYBbKwQQ1PQGpi_W6VN
!unzip -q chest_xray_compact.zip
   
print("✓ Dataset ready!")

In [ ]:
# SOLUTION: Explore dataset structure

data_dir = Path('chest_xray_compact')

# Count images in each split
def count_images(path):
    """Count image files in a directory"""
    return len(list(path.glob('*.jpeg'))) + len(list(path.glob('*.png')))

# Training set
train_normal_path = data_dir / 'train' / 'NORMAL'
train_pneumonia_path = data_dir / 'train' / 'PNEUMONIA'
train_normal_count = count_images(train_normal_path)
train_pneumonia_count = count_images(train_pneumonia_path)

# Validation set
val_normal_count = count_images(data_dir / 'val' / 'NORMAL')
val_pneumonia_count = count_images(data_dir / 'val' / 'PNEUMONIA')

# Test set
test_normal_count = count_images(data_dir / 'test' / 'NORMAL')
test_pneumonia_count = count_images(data_dir / 'test' / 'PNEUMONIA')

print("Dataset Structure:")
print("="*60)
print(f"Training set:")
print(f"  Normal: {train_normal_count} images")
print(f"  Pneumonia: {train_pneumonia_count} images")
print(f"  Total: {train_normal_count + train_pneumonia_count} images")
print(f"\nValidation set:")
print(f"  Normal: {val_normal_count} images")
print(f"  Pneumonia: {val_pneumonia_count} images")
print(f"  Total: {val_normal_count + val_pneumonia_count} images")
print(f"\nTest set:")
print(f"  Normal: {test_normal_count} images")
print(f"  Pneumonia: {test_pneumonia_count} images")
print(f"  Total: {test_normal_count + test_pneumonia_count} images")

if train_normal_count + train_pneumonia_count > 0:
    imbalance_ratio = train_pneumonia_count / (train_normal_count + train_pneumonia_count)
    print(f"\nClass balance: {imbalance_ratio*100:.1f}% pneumonia")
    if imbalance_ratio > 0.7 or imbalance_ratio < 0.3:
        print("⚠️ Significant class imbalance - consider weighted loss!")

In [ ]:
# SOLUTION: Visualize sample images

# Get sample paths
normal_images = list(train_normal_path.glob('*.jpeg')) + list(train_normal_path.glob('*.png'))
pneumonia_images = list(train_pneumonia_path.glob('*.jpeg')) + list(train_pneumonia_path.glob('*.png'))

if len(normal_images) > 0 and len(pneumonia_images) > 0:
    # Load sample images
    normal_img = Image.open(normal_images[0])
    pneumonia_img = Image.open(pneumonia_images[0])
    
    # Display side by side
    fig, axes = plt.subplots(1, 2, figsize=(14, 7))
    
    axes[0].imshow(normal_img, cmap='gray')
    axes[0].set_title('Normal Chest X-Ray', fontsize=16, fontweight='bold', color='#22c55e')
    axes[0].axis('off')
    axes[0].text(0.5, -0.1, f'Size: {normal_img.size}', 
                ha='center', transform=axes[0].transAxes, fontsize=12)
    
    axes[1].imshow(pneumonia_img, cmap='gray')
    axes[1].set_title('Pneumonia Chest X-Ray', fontsize=16, fontweight='bold', color='#ff6b35')
    axes[1].axis('off')
    axes[1].text(0.5, -0.1, f'Size: {pneumonia_img.size}', 
                ha='center', transform=axes[1].transAxes, fontsize=12)
    
    plt.suptitle('Chest X-Ray Comparison', fontsize=18, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()
    
    print("\n👀 What to look for:")
    print("- Normal: Clear lung fields, sharp heart borders")
    print("- Pneumonia: Opacities (white patches), blurred areas")
    print("\nNote: These subtle differences are what the network must learn!")
else:
    print("⚠️ No images found - please ensure dataset is downloaded")

In [ ]:
# SOLUTION: Custom Dataset class

class ChestXRayDataset(Dataset):
    """
    Custom Dataset for Chest X-Ray images
    
    Directory structure:
    root_dir/
        NORMAL/
            image1.jpeg
            image2.jpeg
        PNEUMONIA/
            image1.jpeg
            image2.jpeg
    """
    
    def __init__(self, root_dir, transform=None):
        """
        Args:
            root_dir (str/Path): Directory with NORMAL and PNEUMONIA subdirectories
            transform: Optional transform to be applied on a sample
        """
        self.root_dir = Path(root_dir)
        self.transform = transform
        self.images = []  # List of (image_path, label) tuples
        
        # Collect Normal images (label = 0)
        normal_dir = self.root_dir / 'NORMAL'
        if normal_dir.exists():
            for img_path in normal_dir.glob('*.jpeg'):
                self.images.append((img_path, 0))
            for img_path in normal_dir.glob('*.png'):
                self.images.append((img_path, 0))
        
        # Collect Pneumonia images (label = 1)
        pneumonia_dir = self.root_dir / 'PNEUMONIA'
        if pneumonia_dir.exists():
            for img_path in pneumonia_dir.glob('*.jpeg'):
                self.images.append((img_path, 1))
            for img_path in pneumonia_dir.glob('*.png'):
                self.images.append((img_path, 1))
        
        print(f"Loaded {len(self.images)} images from {root_dir}")
        if len(self.images) > 0:
            labels = [label for _, label in self.images]
            print(f"  Normal: {labels.count(0)} ({labels.count(0)/len(labels)*100:.1f}%)")
            print(f"  Pneumonia: {labels.count(1)} ({labels.count(1)/len(labels)*100:.1f}%)")
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        img_path, label = self.images[idx]
        
        # Load image
        image = Image.open(img_path)
        
        # Convert to RGB (in case image is grayscale or has alpha channel)
        if image.mode != 'RGB':
            image = image.convert('RGB')
        
        # Apply transforms
        if self.transform:
            image = self.transform(image)
        
        return image, label

# Test the dataset class
print("\nTesting ChestXRayDataset class:")
print("="*60)
test_dataset = ChestXRayDataset(data_dir / 'train')
print(f"\nDataset ready with {len(test_dataset)} images")

In [ ]:
# SOLUTION: Define image transforms

class PixelShuffle(object):
    def __init__(self, img_size=(64, 64)):
        self.num_pixels = img_size[0] * img_size[1]
        # Generate the fixed map once upon initialization
        self.perm = torch.randperm(self.num_pixels)

    def __call__(self, tensor):
        # tensor shape: (C, H, W)
        c, h, w = tensor.shape
        # Flatten -> Shuffle -> Reshape
        tensor = tensor.view(c, -1)
        tensor = tensor[:, self.perm]
        return tensor.view(c, h, w)

# Image preprocessing pipeline
transform = transforms.Compose([
    transforms.Resize((64, 64)),           # Resize to 64x64 (smaller for faster training)
    transforms.Grayscale(num_output_channels=1),  # Convert to grayscale
    transforms.ToTensor(),                 # Convert to tensor [0, 1]
    PixelShuffle(),                        # Turn on/off shuffling to see what happens
    transforms.Normalize(mean=[0.5], std=[0.5])  # Normalize to [-1, 1]
])

print("Image Transformation Pipeline:")
print("="*60)
print("1. Resize to 64×64 pixels")
print("   Why: Reduce computation, standardize input size")
print("\n2. Convert to grayscale")
print("   Why: X-rays are inherently grayscale, reduces input size")
print("\n3. Convert to PyTorch tensor")
print("   Why: Neural networks work with tensors")
print("\n4. Normalize to [-1, 1]")
print("   Why: Helps training stability and convergence")
print("\nFinal output: Tensor of shape (1, 64, 64)")
print("              = 1 channel × 64 rows × 64 cols")
print("              = 4,096 pixels total")

In [ ]:
# SOLUTION: Create DataLoaders

# Create datasets
train_dataset = ChestXRayDataset(data_dir / 'train', transform=transform)
val_dataset = ChestXRayDataset(data_dir / 'val', transform=transform)
test_dataset = ChestXRayDataset(data_dir / 'test', transform=transform)

# Create DataLoaders
batch_size = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,      # Shuffle training data
    num_workers=2,     # Parallel data loading
    pin_memory=True    # Faster GPU transfer
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,     # Don't shuffle validation
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,     # Don't shuffle test
    num_workers=2,
    pin_memory=True
)

print("\nDataLoaders Created:")
print("="*60)
print(f"Training:   {len(train_loader)} batches × {batch_size} = {len(train_dataset)} images")
print(f"Validation: {len(val_loader)} batches × {batch_size} ≈ {len(val_dataset)} images")
print(f"Testing:    {len(test_loader)} batches × {batch_size} ≈ {len(test_dataset)} images")

# Test loading a batch
if len(train_loader) > 0:
    images, labels = next(iter(train_loader))
    print(f"\nSample Batch:")
    print(f"  Images shape: {images.shape}  # (batch, channels, height, width)")
    print(f"  Labels shape: {labels.shape}  # (batch,)")
    print(f"  Image value range: [{images.min():.2f}, {images.max():.2f}]")
    print(f"  Labels in batch: {labels.tolist()[:10]}...")

# PART 2: Build MLP Classifier - SOLUTION

In [ ]:
# SOLUTION: Visualize flattening

if len(train_loader) > 0:
    sample_images, sample_labels = next(iter(train_loader))
    sample_img = sample_images[0]  # Shape: (1, 64, 64)
    
    # Flatten
    flattened = sample_img.view(-1)  # Shape: (4096,)
    
    # Visualize
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    
    # Original 2D image
    axes[0].imshow(sample_img.squeeze().numpy(), cmap='gray')
    axes[0].set_title('Original 2D Image\n(64×64 = 4,096 pixels)', 
                     fontsize=14, fontweight='bold')
    axes[0].axis('off')
    axes[0].text(0.5, -0.1, 'Spatial structure preserved', 
                ha='center', transform=axes[0].transAxes, fontsize=11, color='green')
    
    # Flattened 1D vector
    axes[1].plot(flattened.numpy(), linewidth=0.5)
    axes[1].set_title('Flattened to 1D Vector\n(4,096 sequential values)', 
                     fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Pixel Index', fontsize=11)
    axes[1].set_ylabel('Pixel Value', fontsize=11)
    axes[1].grid(True, alpha=0.3)
    axes[1].text(0.5, -0.15, 'Spatial structure destroyed!', 
                ha='center', transform=axes[1].transAxes, fontsize=11, color='red')
    
    # Explanation
    explanation = (
        "⚠️ CRITICAL ISSUE:\n\n"
        "Pixels that were neighbors\n"
        "in the 2D image are now\n"
        "scattered in the 1D vector!\n\n"
        "Example:\n"
        "• Pixel [10, 10] and [10, 11]\n"
        "  were adjacent (1 pixel apart)\n\n"
        "• After flattening:\n"
        "  Index 650 and 651\n"
        "  (still close)\n\n"
        "• But pixel [10, 10] and [11, 10]\n"
        "  were also adjacent\n\n"
        "• After flattening:\n"
        "  Index 650 and 714\n"
        "  (64 positions apart!)\n\n"
        "MLP treats all inputs\n"
        "independently - it doesn't\n"
        "know about spatial proximity!"
    )
    axes[2].text(0.05, 0.95, explanation,
                ha='left', va='top', fontsize=9,
                transform=axes[2].transAxes,
                bbox=dict(boxstyle='round', facecolor='#fff4e6', alpha=0.9, edgecolor='#ff6b35', linewidth=2))
    axes[2].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    print("\n📊 Shape Transformation:")
    print("="*60)
    print(f"Original: {sample_img.shape} → 2D structure maintained")
    print(f"Flattened: {flattened.shape} → All spatial info lost!")
    
    print("\n🤔 Critical Question for Students:")
    print("Does the MLP know that pixel 650 and pixel 714 were neighbors?")
    print("Answer: NO! It treats them as independent features.")
    print("\nThis is why MLPs struggle with images!")

In [ ]:
# SOLUTION: MLP Architecture for Images

class ChestXRayMLP(nn.Module):
    """
    Multi-Layer Perceptron for Chest X-Ray Classification
    
    Architecture:
        Input: 4096 pixels (64×64 flattened)
        Hidden1: 128 neurons + ReLU
        Hidden2: 64 neurons + ReLU  
        Output: 2 classes (Normal vs Pneumonia)
    
    Note: This is deliberately simple to show limitations!
    """
    
    def __init__(self, input_size=4096, hidden1=128, hidden2=64, num_classes=2):
        super(ChestXRayMLP, self).__init__()
        
        # Define layers
        self.fc1 = nn.Linear(input_size, hidden1)  # 4096 → 128
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden1, hidden2)      # 128 → 64
        self.fc3 = nn.Linear(hidden2, num_classes)  # 64 → 2
        
    def forward(self, x):
        # Flatten image: (batch, 1, 64, 64) → (batch, 4096)
        x = x.view(x.size(0), -1)
        
        # Forward pass
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.relu(x)
        x = self.fc3(x)  # No activation - CrossEntropyLoss includes softmax
        
        return x

# Create model
model = ChestXRayMLP()

print("Model Architecture:")
print("="*60)
print(model)
print("="*60)

# Count parameters
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

total_params = count_parameters(model)
print(f"\nTotal Trainable Parameters: {total_params:,}")
print("\nParameter Breakdown:")
print(f"  fc1: (4096 × 128) + 128 bias = {4096*128 + 128:,}")
print(f"  fc2: (128 × 64) + 64 bias = {128*64 + 64:,}")
print(f"  fc3: (64 × 2) + 2 bias = {64*2 + 2:,}")
print(f"  Total: {total_params:,}")

print("\n📊 Comparison to Lab 2:")
print(f"  Lab 2 (diabetes): ~300 parameters")
print(f"  Lab 3 (X-rays): {total_params:,} parameters")
print(f"  Ratio: {total_params/300:.0f}× larger!")
print("\nWhy? Input size: 8 features vs 4,096 pixels")

# Test forward pass
test_input = torch.randn(1, 1, 64, 64)
test_output = model(test_input)
print(f"\n✓ Forward pass test:")
print(f"  Input: {test_input.shape} → Output: {test_output.shape}")
print(f"  Expected: (1, 2) ✓")

# PART 3: Training & Evaluation - SOLUTION

In [ ]:
# SOLUTION: Training setup

# Loss function
criterion = nn.CrossEntropyLoss()

# Optimizer
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

# Training parameters
num_epochs = 10  # Fewer than Lab 2 due to slower training

print("Training Configuration:")
print("="*60)
print(f"Device: {device}")
print(f"Loss Function: CrossEntropyLoss")
print(f"Optimizer: Adam(lr=0.001)")
print(f"Batch Size: {batch_size}")
print(f"Epochs: {num_epochs}")
print(f"\nModel on device: {next(model.parameters()).device}")
print("="*60)

In [ ]:
# SOLUTION: Training loop

# Storage for metrics
train_losses = []
val_losses = []
val_accuracies = []
epoch_times = []

print("Starting Training...\n")
print("Epoch | Train Loss | Val Loss | Val Acc | Time")
print("-" * 55)

for epoch in range(num_epochs):
    epoch_start = time.time()
    
    # ============================================
    # TRAINING PHASE
    # ============================================
    model.train()
    train_loss = 0.0
    
    for batch_idx, (images, labels) in enumerate(train_loader):
        # Move to device
        images = images.to(device)
        labels = labels.to(device)
        
        # Zero gradients
        optimizer.zero_grad()
        
        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward pass
        loss.backward()
        
        # Update weights
        optimizer.step()
        
        train_loss += loss.item()
    
    train_loss /= len(train_loader)
    
    # ============================================
    # VALIDATION PHASE
    # ============================================
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    val_loss /= len(val_loader)
    val_accuracy = 100 * correct / total if total > 0 else 0
    
    # Store metrics
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_accuracies.append(val_accuracy)
    
    epoch_time = time.time() - epoch_start
    epoch_times.append(epoch_time)
    
    # Print progress
    print(f"{epoch+1:5d} | {train_loss:10.4f} | {val_loss:8.4f} | {val_accuracy:6.2f}% | {epoch_time:4.1f}s")

print("\n" + "="*60)
print("Training Complete!")
print("="*60)
print(f"Average time per epoch: {np.mean(epoch_times):.1f}s")
print(f"Total training time: {sum(epoch_times)/60:.1f} minutes")

In [ ]:
# SOLUTION: Visualize training progress

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss curves
axes[0].plot(train_losses, label='Training Loss', linewidth=2, marker='o', color='#0066cc')
axes[0].plot(val_losses, label='Validation Loss', linewidth=2, marker='s', color='#ff6b35')
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Training vs Validation Loss', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Accuracy curve
axes[1].plot(val_accuracies, label='Validation Accuracy', linewidth=2, marker='D', color='#22c55e')
axes[1].axhline(y=50, color='red', linestyle='--', linewidth=2, label='Random Guess (50%)', alpha=0.7)
axes[1].axhline(y=70, color='orange', linestyle='--', linewidth=2, label='Target (70%)', alpha=0.7)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Accuracy (%)', fontsize=12)
axes[1].set_title('Validation Accuracy', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim([0, 100])

# Training time per epoch
axes[2].bar(range(1, len(epoch_times)+1), epoch_times, color='#9333ea', alpha=0.7)
axes[2].axhline(y=np.mean(epoch_times), color='red', linestyle='--', linewidth=2, label=f'Average: {np.mean(epoch_times):.1f}s')
axes[2].set_xlabel('Epoch', fontsize=12)
axes[2].set_ylabel('Time (seconds)', fontsize=12)
axes[2].set_title('Training Time per Epoch', fontsize=14, fontweight='bold')
axes[2].legend(fontsize=11)
axes[2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\nFinal Results:")
print("="*60)
print(f"Best Validation Accuracy: {max(val_accuracies):.2f}% (epoch {val_accuracies.index(max(val_accuracies))+1})")
print(f"Final Validation Accuracy: {val_accuracies[-1]:.2f}%")
print(f"Final Training Loss: {train_losses[-1]:.4f}")
print(f"Final Validation Loss: {val_losses[-1]:.4f}")

# Analysis
if train_losses[-1] < val_losses[-1] * 0.7:
    print("\n⚠️ Overfitting detected: Train loss << Val loss")
elif val_accuracies[-1] < 60:
    print("\n⚠️ Poor performance: Accuracy below 60%")
    print("   This is expected with MLP on images!")
elif val_accuracies[-1] > 75:
    print("\n✓ Good performance for an MLP!")
else:
    print("\n✓ Moderate performance - typical for MLP on images")

In [ ]:
# SOLUTION: Evaluate on test set

model.eval()

all_preds = []
all_labels = []
all_probs = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)
        _, predicted = torch.max(outputs, 1)
        
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs[:, 1].cpu().numpy())  # Probability of pneumonia

# Convert to numpy
all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs = np.array(all_probs)

# Compute metrics
accuracy = accuracy_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds)
sensitivity = recall_score(all_labels, all_preds)  # Recall = Sensitivity

# Specificity (manually)
cm = confusion_matrix(all_labels, all_preds)
tn, fp, fn, tp = cm.ravel()
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

# Print results
print("\n" + "="*60)
print(" "*15 + "TEST SET RESULTS")
print("="*60)
print(f"Accuracy:    {accuracy*100:.2f}%")
print(f"Precision:   {precision*100:.2f}%  (Of predicted pneumonia, how many correct?)")
print(f"Sensitivity: {sensitivity*100:.2f}%  (Of actual pneumonia, how many caught?)")
print(f"Specificity: {specificity*100:.2f}%  (Of actual normal, how many correct?)")
print("="*60)

# Comparison to baseline
baseline = max(np.mean(all_labels), 1 - np.mean(all_labels))
print(f"\nBaseline (always predict majority): {baseline*100:.2f}%")
print(f"Our model: {accuracy*100:.2f}%")
print(f"Improvement: +{(accuracy - baseline)*100:.2f}%")

# Detailed classification report
print("\nDetailed Classification Report:")
print("="*60)
print(classification_report(all_labels, all_preds, 
                          target_names=['Normal', 'Pneumonia'],
                          digits=3))

In [ ]:
# SOLUTION: Confusion matrix visualization

plt.figure(figsize=(10, 8))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal', 'Pneumonia'],
            yticklabels=['Normal', 'Pneumonia'],
            cbar_kws={'label': 'Count'},
            annot_kws={'size': 16, 'weight': 'bold'})

plt.ylabel('True Label', fontsize=14, fontweight='bold')
plt.xlabel('Predicted Label', fontsize=14, fontweight='bold')
plt.title('Confusion Matrix - Pneumonia Detection\n', fontsize=16, fontweight='bold')

# Add interpretation
interpretation = f"""
True Negatives (TN): {tn} - Correctly identified normal X-rays
False Positives (FP): {fp} - Normal incorrectly labeled as pneumonia
False Negatives (FN): {fn} - Pneumonia cases MISSED (dangerous!)
True Positives (TP): {tp} - Correctly identified pneumonia

Sensitivity: {sensitivity*100:.1f}% - We caught {tp}/{tp+fn} pneumonia cases
Specificity: {specificity*100:.1f}% - We correctly identified {tn}/{tn+fp} normal cases
"""

plt.text(0.5, -0.25, interpretation,
        ha='center', transform=plt.gca().transAxes,
        fontsize=11,
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

plt.tight_layout()
plt.show()

print("\nClinical Implications:")
print("="*60)
if sensitivity < 0.70:
    print("⚠️ LOW SENSITIVITY: Missing too many pneumonia cases!")
    print(f"   We're only catching {sensitivity*100:.1f}% of pneumonia patients.")
    print("   In a real clinical setting, this would be unacceptable.")
elif sensitivity > 0.85:
    print("✓ GOOD SENSITIVITY: Catching most pneumonia cases")
else:
    print("✓ ACCEPTABLE SENSITIVITY for a basic model")
    print(f"   Catching {sensitivity*100:.1f}% of pneumonia cases.")
    print("   Room for improvement with better architecture!")

# PART 4: Reflection & Analysis - SOLUTION

In [ ]:
# SOLUTION: Performance comparison visualization

# Create comparison table
comparison_data = {
    'Metric': ['Input Features', 'Parameters', 'Accuracy', 'Training Time/Epoch'],
    'Lab 2 (Diabetes)': ['8', '~300', '~76%', '~2s'],
    'Lab 3 (X-Ray)': ['4,096', f'{total_params:,}', f'{accuracy*100:.1f}%', f'{np.mean(epoch_times):.1f}s'],
    'Ratio': ['512×', f'{total_params/300:.0f}×', f'{accuracy/0.76:.2f}×', f'{np.mean(epoch_times)/2:.1f}×']
}

comparison_df = pd.DataFrame(comparison_data)

print("\n" + "="*70)
print(" "*20 + "LAB 2 vs LAB 3 COMPARISON")
print("="*70)
print(comparison_df.to_string(index=False))
print("="*70)

print("\n🤔 KEY OBSERVATIONS:")
print("-" * 70)
print("1. MANY MORE inputs (4,096 vs 8) - 512× increase")
print("2. MUCH LARGER model (500K vs 300 params) - 1,600× increase")
print(f"3. BUT... {'' if accuracy < 0.76 else 'sur'}prisingly {'WORSE' if accuracy < 0.76 else 'similar'} accuracy!")
print("4. SLOWER training (due to larger model)")

print("\n💡 WHY IS THIS HAPPENING?")
print("-" * 70)
print("The problem is NOT the amount of data or model size.")
print("The problem is the ARCHITECTURE!")
print("\nMLPs treat each pixel independently.")
print("They don't understand that nearby pixels form patterns (edges, shapes).")
print("\nIn medical images, SPATIAL PATTERNS matter:")
print("  • Lung opacities (clusters of bright pixels)")
print("  • Heart borders (edges)")
print("  • Organ positions (relative locations)")
print("\nFlattening the image DESTROYS this spatial information!")

print("\n🎯 WHAT WE NEED:")
print("-" * 70)
print("An architecture that:")
print("  1. Processes nearby pixels together")
print("  2. Looks for local patterns (edges, textures)")
print("  3. Understands that shifting an image doesn't change the diagnosis")
print("  4. Builds hierarchical understanding (pixels → edges → shapes → organs)")
print("\n👉 Coming soon: Convolutional Neural Networks (CNNs)!")

## Sample Student Reflection Answers

### Question 1: Performance Comparison

**Why might we get WORSE accuracy despite having MORE features and a BIGGER model?**

**Expected student answers:**
- "The model has too many parameters and not enough data (overfitting)"
- "Flattening loses important spatial information"
- "The network doesn't know which pixels are neighbors"
- "Wrong architecture for the type of data"

**Teaching moment:** Emphasize that architecture matters MORE than size!

---

### Question 2: The Flattening Problem

**1. Does the MLP know that pixel 0 and pixel 1 were next to each other?**

**Answer:** NO! After flattening, all pixels are just independent numbers. The network has no concept of spatial proximity.

**2. In a chest X-ray, what matters more: individual pixel values OR patterns of nearby pixels?**

**Answer:** Patterns! Radiologists look for shapes, textures, and relative positions - not individual pixel values.

**3. Does our MLP naturally detect clusters?**

**Answer:** NO! It would need to learn complex interactions between specific pixel positions, which is very difficult.

---

### Question 3: What's Missing?

**Medical information lost by flattening:**
- Shape of organs (heart silhouette, lung boundaries)
- Texture patterns (smooth vs patchy)
- Edge information (sharp vs blurry boundaries)
- Relative positions (upper vs lower lobe)
- Symmetry (comparing left vs right lung)

---

### Question 4: Better Architecture Ideas

**Expected student ideas:**
- "Process small regions of the image at a time"
- "Look for patterns like edges first, then combine them"
- "Use filters that scan across the image"
- "Build understanding hierarchically - small details first, then bigger structures"
- "Make the network understand that the same pattern anywhere in the image means the same thing"

**Perfect!** These are exactly the ideas behind CNNs!

---

## 🎓 Teaching Notes & Discussion Points

### Expected Student Reactions:

✅ **"This was harder than Lab 2"** - Perfect! That's the point.

✅ **"Why isn't it working better?"** - Excellent question! Architecture matters.

✅ **"Can we use a different type of network?"** - YES! CNNs coming soon.

✅ **"This is frustrating"** - Productive struggle! Now they'll appreciate CNNs.

## Summary

This lab successfully:
1. ✅ Applied Lab 2 knowledge to new domain
2. ✅ Demonstrated MLP limitations on images
3. ✅ Created motivation for CNNs

**The "productive struggle" is the pedagogical WIN!** 🎯

---